# Diabetic Retinopathy — Custom CNN (from scratch)

**Track:** Custom (no pretrained weights)

**Metric:** AUC (area under ROC curve)

**Submission:** `output_custom.csv` — 1000×1 file of DR scores, zipped to `codabench_submission.zip`

This notebook is self-contained. Run every cell top-to-bottom to reproduce results and generate the submission file.

In [ ]:
%matplotlib inline
from __future__ import print_function, division
import os
import time
import copy
import random
import csv
from zipfile import ZipFile

import numpy as np
import numpy.random as npr
import pandas as pd
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.optim import lr_scheduler
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms

from skimage import io, transform, util, color
from sklearn import metrics
import cv2
from PIL import Image

import warnings
warnings.filterwarnings('ignore')

# ── Reproducibility ──────────────────────────────────────────────────────────
random.seed(42)
npr.seed(42)
torch.manual_seed(42)
torch.backends.cudnn.enabled = False

device = torch.device('cuda:0' if torch.cuda.is_available() else 'cpu')
print('Device:', device)

## Dataset class and transforms

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# Dataset
# ─────────────────────────────────────────────────────────────────────────────

class RetinopathyDataset(Dataset):
    """Retinopathy dataset."""

    def __init__(self, csv_file, root_dir, transform=None, maxSize=0):
        self.dataset = pd.read_csv(csv_file, header=0,
                                   dtype={'id': str, 'eye': int, 'label': int})
        if maxSize > 0:
            idx = np.random.RandomState(seed=42).permutation(range(len(self.dataset)))
            self.dataset = self.dataset.iloc[idx[:maxSize]].reset_index(drop=True)
        self.root_dir = root_dir
        self.img_dir  = os.path.join(root_dir, 'images')
        self.transform = transform

    def __len__(self):
        return len(self.dataset)

    def __getitem__(self, idx):
        if torch.is_tensor(idx):
            idx = idx.tolist()
        img_name = os.path.join(self.img_dir, self.dataset.id[idx] + '.jpg')
        image = io.imread(img_name)
        if self.dataset.eye[idx] == 1:          # mirror right-eye images
            image = image[:, ::-1, :]
        sample = {
            'image': image,
            'eye':   self.dataset.eye[idx],
            'label': (self.dataset.label[idx] > 0).astype(dtype=np.int64),
        }
        if self.transform:
            sample = self.transform(sample)
        return sample


# ─────────────────────────────────────────────────────────────────────────────
# Custom transform classes (operate on sample dicts)
# ─────────────────────────────────────────────────────────────────────────────

class CropByEye(object):
    """Crop tightly around the fundus using a brightness threshold."""
    def __init__(self, threshold, border):
        self.threshold = threshold
        self.border = (border, border) if isinstance(border, int) else border

    def __call__(self, sample):
        image, eye, label = sample['image'], sample['eye'], sample['label']
        h, w = image.shape[:2]
        imgray = color.rgb2gray(image)
        _, mask = cv2.threshold(imgray, self.threshold, 1, cv2.THRESH_BINARY)
        sidx = np.nonzero(mask)
        if len(sidx[0]) < 20:
            return {'image': image, 'eye': eye, 'label': label}
        minx = np.maximum(sidx[1].min() - self.border[1], 0)
        maxx = np.minimum(sidx[1].max() + 1 + self.border[1], w)
        miny = np.maximum(sidx[0].min() - self.border[0], 0)
        maxy = np.minimum(sidx[0].max() + 1 + self.border[1], h)
        image = image[miny:maxy, minx:maxx, ...]
        return {'image': image, 'eye': eye, 'label': label}


class Rescale(object):
    """Resize so the shortest side equals output_size (int) or exact (h,w) tuple."""
    def __init__(self, output_size):
        self.output_size = output_size

    def __call__(self, sample):
        image, eye, label = sample['image'], sample['eye'], sample['label']
        h, w = image.shape[:2]
        if isinstance(self.output_size, int):
            new_h, new_w = ((self.output_size * h / w, self.output_size)
                            if h > w else
                            (self.output_size, self.output_size * w / h))
        else:
            new_h, new_w = self.output_size
        image = transform.resize(image, (int(new_h), int(new_w)))
        return {'image': image, 'eye': eye, 'label': label}


class RandomCrop(object):
    """Randomly crop to output_size (int for square, or (h,w) tuple)."""
    def __init__(self, output_size):
        self.output_size = ((output_size, output_size)
                            if isinstance(output_size, int) else output_size)

    def __call__(self, sample):
        image, eye, label = sample['image'], sample['eye'], sample['label']
        h, w = image.shape[:2]
        new_h, new_w = self.output_size
        top  = np.random.randint(0, h - new_h) if h > new_h else 0
        left = np.random.randint(0, w - new_w) if w > new_w else 0
        image = image[top:top + new_h, left:left + new_w]
        return {'image': image, 'eye': eye, 'label': label}


class CenterCrop(object):
    """Center-crop to output_size (int for square, or (h,w) tuple)."""
    def __init__(self, output_size):
        self.output_size = ((output_size, output_size)
                            if isinstance(output_size, int) else output_size)

    def __call__(self, sample):
        image, eye, label = sample['image'], sample['eye'], sample['label']
        h, w = image.shape[:2]
        new_h, new_w = self.output_size
        top  = int((h - new_h) / 2) if h > new_h else 0
        left = int((w - new_w) / 2) if w > new_w else 0
        image = image[top:top + new_h, left:left + new_w]
        return {'image': image, 'eye': eye, 'label': label}


class ToTensor(object):
    """numpy HxWxC float64 → torch CxHxW; label → long tensor."""
    def __call__(self, sample):
        image, eye, label = sample['image'], sample['eye'], sample['label']
        image = torch.from_numpy(image.transpose((2, 0, 1)))
        label = torch.tensor(label, dtype=torch.long)
        return {'image': image, 'eye': eye, 'label': label}


class Normalize(object):
    """Per-channel normalization on a tensor image."""
    def __init__(self, mean, std):
        self.mean = np.array(mean)
        self.std  = np.array(std)

    def __call__(self, sample):
        image, eye, label = sample['image'], sample['eye'], sample['label']
        dtype = image.dtype
        mean = torch.as_tensor(self.mean, dtype=dtype, device=image.device)
        std  = torch.as_tensor(self.std,  dtype=dtype, device=image.device)
        image.sub_(mean[:, None, None]).div_(std[:, None, None])
        return {'image': image, 'eye': eye, 'label': label}


# ─────────────────────────────────────────────────────────────────────────────
# torchvision-backed wrappers (numpy float64 → PIL → TV transform → numpy float64)
# ─────────────────────────────────────────────────────────────────────────────

class TVRandomHorizontalFlip(object):
    """Stochastic horizontal flip (p=0.5 by default)."""
    def __init__(self, p=0.5):
        self.RHF = transforms.RandomHorizontalFlip(p=p)
    def __call__(self, sample):
        image, eye, label = sample['image'], sample['eye'], sample['label']
        pil = Image.fromarray(util.img_as_ubyte(image))
        image = util.img_as_float(np.asarray(self.RHF(pil)))
        return {'image': image, 'eye': eye, 'label': label}


class TVRandomRotation(object):
    """Random rotation in [-degrees, +degrees]. Safe for fundus images (rotationally invariant)."""
    def __init__(self, degrees):
        self.RR = transforms.RandomRotation(degrees)
    def __call__(self, sample):
        image, eye, label = sample['image'], sample['eye'], sample['label']
        pil = Image.fromarray(util.img_as_ubyte(image))
        image = util.img_as_float(np.asarray(self.RR(pil)))
        return {'image': image, 'eye': eye, 'label': label}


class TVColorJitter(object):
    """Random brightness/contrast/saturation/hue jitter. Models device variability across clinical sites."""
    def __init__(self, brightness=0.2, contrast=0.2, saturation=0.1, hue=0.05):
        self.CJ = transforms.ColorJitter(brightness=brightness, contrast=contrast,
                                          saturation=saturation, hue=hue)
    def __call__(self, sample):
        image, eye, label = sample['image'], sample['eye'], sample['label']
        pil = Image.fromarray(util.img_as_ubyte(image))
        image = util.img_as_float(np.asarray(self.CJ(pil)))
        return {'image': image, 'eye': eye, 'label': label}


print('All classes defined.')

## Datasets and DataLoaders

In [ ]:
pixel_mean = [0.485, 0.456, 0.406]   # ImageNet channel means
pixel_std  = [0.229, 0.224, 0.225]   # ImageNet channel stds

# ── Train: full dataset (maxSize=0), with data augmentation ──────────────────
train_dataset = RetinopathyDataset(
    csv_file='data/train.csv',
    root_dir='data',
    maxSize=0,    # 0 = use all 2000 samples; set to e.g. 200 for quick dev runs
    transform=transforms.Compose([
        CropByEye(0.10, 1),
        Rescale(256),               # rescale to 256 so RandomCrop(224) always fits
        RandomCrop(224),
        TVRandomHorizontalFlip(p=0.5),
        TVRandomRotation(15),
        TVColorJitter(brightness=0.2, contrast=0.2, saturation=0.1, hue=0.05),
        ToTensor(),
        Normalize(mean=pixel_mean, std=pixel_std),
    ])
)

# ── Val: deterministic pipeline ───────────────────────────────────────────────
val_dataset = RetinopathyDataset(
    csv_file='data/val.csv',
    root_dir='data',
    transform=transforms.Compose([
        CropByEye(0.10, 1),
        Rescale(256),
        CenterCrop(224),
        ToTensor(),
        Normalize(mean=pixel_mean, std=pixel_std),
    ])
)

# ── Test: same deterministic pipeline ────────────────────────────────────────
test_dataset = RetinopathyDataset(
    csv_file='data/test.csv',
    root_dir='data',
    transform=transforms.Compose([
        CropByEye(0.10, 1),
        Rescale(256),
        CenterCrop(224),
        ToTensor(),
        Normalize(mean=pixel_mean, std=pixel_std),
    ])
)

# ── DataLoaders (num_workers=0 avoids forkserver pickling issues in Jupyter) ──
train_dataloader = DataLoader(train_dataset, batch_size=32, shuffle=True,  num_workers=0)
val_dataloader   = DataLoader(val_dataset,   batch_size=64, shuffle=False, num_workers=0)
test_dataloader  = DataLoader(test_dataset,  batch_size=64, shuffle=False, num_workers=0)

print(f'Train: {len(train_dataset)} | Val: {len(val_dataset)} | Test: {len(test_dataset)}')

# Required by train_model / test_model
image_datasets = {'train': train_dataset, 'val': val_dataset}
dataloaders    = {'train': train_dataloader, 'val': val_dataloader}
dataset_sizes  = {'train': len(train_dataset), 'val': len(val_dataset)}

## Custom CNN architecture

VGG-inspired design: four convolutional blocks (3×3 kernels, BatchNorm, ReLU, MaxPool),
followed by Global Average Pooling and two fully-connected layers with Dropout.

| Stage | Output shape |
|---|---|
| Input | B × 3 × 224 × 224 |
| Block 1: conv(3→32)×2 + pool | B × 32 × 112 × 112 |
| Block 2: conv(32→64)×2 + pool | B × 64 × 56 × 56 |
| Block 3: conv(64→128)×2 + pool | B × 128 × 28 × 28 |
| Block 4: conv(128→256) + pool | B × 256 × 14 × 14 |
| AdaptiveAvgPool2d(1) + flatten | B × 256 |
| Dropout(0.5) → Linear(256→128) → ReLU | B × 128 |
| Dropout(0.3) → Linear(128→1) | B × 1 (raw logit) |

The output is a raw logit — **no sigmoid in `forward()`**.
`BCEWithLogitsLoss` applies sigmoid internally (numerically stable).
Sigmoid is applied explicitly only for AUC evaluation and test inference.

In [ ]:
class CustomNet(nn.Module):
    def __init__(self):
        super().__init__()

        def _block(in_ch, out_ch):
            """Conv3x3 + BatchNorm + ReLU (bias=False because BN absorbs it)."""
            return nn.Sequential(
                nn.Conv2d(in_ch, out_ch, kernel_size=3, padding=1, bias=False),
                nn.BatchNorm2d(out_ch),
                nn.ReLU(inplace=True),
            )

        # ── Convolutional backbone ────────────────────────────────────────────
        self.block1 = nn.Sequential(
            _block(3,  32), _block(32, 32), nn.MaxPool2d(2, 2))
        self.block2 = nn.Sequential(
            _block(32, 64), _block(64, 64), nn.MaxPool2d(2, 2))
        self.block3 = nn.Sequential(
            _block(64, 128), _block(128, 128), nn.MaxPool2d(2, 2))
        self.block4 = nn.Sequential(
            _block(128, 256), nn.MaxPool2d(2, 2))

        # ── Global average pooling collapses spatial dims to 1×1 ─────────────
        self.gap = nn.AdaptiveAvgPool2d(1)

        # ── Classifier head ───────────────────────────────────────────────────
        self.classifier = nn.Sequential(
            nn.Dropout(0.5),
            nn.Linear(256, 128),
            nn.ReLU(inplace=True),
            nn.Dropout(0.3),
            nn.Linear(128, 1),   # raw logit — sigmoid applied externally
        )

    def forward(self, x):
        x = self.block1(x)
        x = self.block2(x)
        x = self.block3(x)
        x = self.block4(x)
        x = self.gap(x)
        x = x.flatten(1)        # B×256×1×1 → B×256
        x = self.classifier(x)  # B×256 → B×1
        return x


# ── Quick shape sanity check ──────────────────────────────────────────────────
with torch.no_grad():
    _dummy = torch.zeros(2, 3, 224, 224)
    _out   = CustomNet()(_dummy)
    assert _out.shape == (2, 1), f'Expected (2,1), got {_out.shape}'
    print('Shape check passed:', _out.shape)

total_params = sum(p.numel() for p in CustomNet().parameters() if p.requires_grad)
print(f'Trainable parameters: {total_params:,}')

## Loss, optimizer, and LR scheduler

In [ ]:
customNet = CustomNet().to(device)

# BCEWithLogitsLoss fuses sigmoid + BCE in one numerically-stable operation.
# Do NOT apply sigmoid in forward() when using this loss.
criterion = nn.BCEWithLogitsLoss()

# Adam converges faster than SGD on small datasets; weight_decay adds mild L2 regularisation.
optimizer_custom = optim.Adam(customNet.parameters(), lr=1e-3, weight_decay=1e-4)

# LR drops ×0.1 at epoch 7 and epoch 14 (3-phase schedule over 20 epochs).
scheduler_custom = lr_scheduler.StepLR(optimizer_custom, step_size=7, gamma=0.1)

## Training function

In [ ]:
def train_model(model, criterion, optimizer, scheduler, num_epochs=20, patience=5):
    """Train with early stopping on val AUC.

    Args:
        model      : nn.Module with raw-logit output (no sigmoid in forward).
        criterion  : BCEWithLogitsLoss.
        optimizer  : any torch optimizer.
        scheduler  : LR scheduler (stepped once per epoch after train phase).
        num_epochs : maximum number of epochs.
        patience   : stop early if val AUC does not improve for this many epochs.

    Returns:
        model loaded with best-epoch weights.
    """
    since = time.time()

    best_model_wts  = copy.deepcopy(model.state_dict())
    best_auc        = 0.0
    best_epoch      = -1
    epochs_no_improve = 0

    for epoch in range(num_epochs):
        print(f'Epoch {epoch}/{num_epochs - 1}')
        print('-' * 10)

        for phase in ['train', 'val']:
            if phase == 'train':
                model.train()
            else:
                model.eval()

            num_samples   = dataset_sizes[phase]
            outputs_m     = np.zeros((num_samples,), dtype=float)
            labels_m      = np.zeros((num_samples,), dtype=int)
            running_loss  = 0.0
            cont_samples  = 0

            for sample in dataloaders[phase]:
                inputs = sample['image'].to(device).float()
                labels = sample['label'].to(device).float()
                batch_size = labels.shape[0]

                optimizer.zero_grad()

                with torch.set_grad_enabled(phase == 'train'):
                    outputs = model(inputs)          # raw logit: (B, 1)
                    logits  = outputs.flatten()      # (B,)

                    # BCEWithLogitsLoss expects logits (no sigmoid before it)
                    loss = criterion(logits, labels)

                    # Explicit sigmoid only for AUC accumulation
                    probs = torch.sigmoid(logits)

                    if phase == 'train':
                        loss.backward()
                        optimizer.step()

                running_loss += loss.item() * inputs.size(0)
                outputs_m[cont_samples:cont_samples + batch_size] = probs.detach().cpu().numpy()
                labels_m [cont_samples:cont_samples + batch_size] = labels.cpu().numpy()
                cont_samples += batch_size

            if phase == 'train':
                scheduler.step()

            epoch_loss = running_loss / dataset_sizes[phase]
            epoch_auc  = metrics.roc_auc_score(labels_m, outputs_m)
            print(f'{phase:5s}  Loss: {epoch_loss:.4f}  AUC: {epoch_auc:.4f}')

            if phase == 'val':
                if epoch_auc > best_auc:
                    best_auc       = epoch_auc
                    best_model_wts = copy.deepcopy(model.state_dict())
                    best_epoch     = epoch
                    epochs_no_improve = 0
                else:
                    epochs_no_improve += 1
                    if epochs_no_improve >= patience:
                        print(f'Early stopping at epoch {epoch} '
                              f'(no val AUC improvement for {patience} epochs).')
                        model.load_state_dict(best_model_wts)
                        elapsed = time.time() - since
                        print(f'Training complete in {elapsed//60:.0f}m {elapsed%60:.0f}s')
                        print(f'Best epoch: {best_epoch}  Best val AUC: {best_auc:.4f}')
                        return model
        print()

    elapsed = time.time() - since
    print(f'Training complete in {elapsed//60:.0f}m {elapsed%60:.0f}s')
    print(f'Best epoch: {best_epoch}  Best val AUC: {best_auc:.4f}')
    model.load_state_dict(best_model_wts)
    return model

## Test inference function

In [ ]:
def test_model(model):
    """Run inference on the test set.

    Returns:
        numpy array of shape (N, 1) with sigmoid DR scores in [0, 1].
    """
    since = time.time()
    model.eval()

    num_samples = len(test_dataset)
    outputs_m   = np.zeros((num_samples, 1), dtype=float)
    cont        = 0

    for sample in test_dataloader:
        inputs     = sample['image'].to(device).float()
        batch_size = inputs.shape[0]
        with torch.no_grad():
            # sigmoid converts raw logit → probability in [0, 1]
            outputs = torch.sigmoid(model(inputs))
            outputs_m[cont:cont + batch_size, :] = outputs.cpu().numpy()
        cont += batch_size

    elapsed = time.time() - since
    print(f'Inference done in {elapsed:.1f}s')
    return outputs_m

## Train

Seeds are fixed immediately before training for full reproducibility.

In [ ]:
random.seed(42)
npr.seed(42)
torch.manual_seed(42)
torch.cuda.manual_seed_all(42)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False

customNet = train_model(
    customNet,
    criterion,
    optimizer_custom,
    scheduler_custom,
    num_epochs=20,
    patience=5,
)

## Final validation AUC (best-weight model)

In [ ]:
customNet.eval()
val_labels, val_probs = [], []
with torch.no_grad():
    for sample in val_dataloader:
        inputs = sample['image'].to(device).float()
        labels = sample['label'].numpy()
        probs  = torch.sigmoid(customNet(inputs)).cpu().numpy().flatten()
        val_probs.extend(probs)
        val_labels.extend(labels)

final_val_auc = metrics.roc_auc_score(val_labels, val_probs)
print(f'Final val AUC (best-weight model): {final_val_auc:.4f}')

## Generate test predictions

In [ ]:
outputs_custom = test_model(customNet)
print('Output shape:', outputs_custom.shape)   # expect (1000, 1)
print('Score range: [{:.4f}, {:.4f}]'.format(outputs_custom.min(), outputs_custom.max()))

## Save CSV and create submission ZIP

In [ ]:
# Save the custom CSV to the root directory, matching the original notebook convention.
with open('output_custom.csv', mode='w') as out_file:
    csv_writer = csv.writer(out_file, delimiter=',')
    csv_writer.writerows(outputs_custom)
print(f'Saved output_custom.csv  ({len(outputs_custom)} rows)')

# ── Submission ZIP ────────────────────────────────────────────────────────────
# The challenge requires one ZIP containing BOTH output_custom.csv and output_ft.csv.
# This notebook only generates the Custom track file. Once you have run the
# fine-tuning notebook and generated output_ft.csv, bundle them together:
#
#   with ZipFile('./codabench_submission.zip', 'w') as z:
#       z.write('./output_custom.csv')
#       z.write('./output_ft.csv')
#
# For a Custom-track-only submission (Codabench allows per-category uploads),
# the ZIP below contains only output_custom.csv:
with ZipFile('./codabench_submission.zip', 'w') as zip_object:
    zip_object.write('./output_custom.csv')
print('Saved codabench_submission.zip')